# Solution D3 · Reading a 10x10 confusion matrix with two rare classes

**Type**: Interpret

## Objective

Interpret the metrics of a multiclass classifier when two classes are rare in the training set: precision, recall and F1 **per class**, micro, macro and weighted average, and understand where
the errors of the rare classes end up. Then fix with `class_weight="balanced"`.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X_tr, X_te, y_tr, y_te = train_test_split(digits.data, digits.target, test_size=0.4, stratify=digits.target, random_state=0)
rng = np.random.default_rng(0)
keep = np.ones(len(y_tr), dtype=bool)
for c in (3, 8):
    idx = np.where(y_tr == c)[0]
    keep[rng.choice(idx, int(0.85 * len(idx)), replace=False)] = False
X_s, y_s = X_tr[keep], y_tr[keep]
print("examples per class in the training set:", np.bincount(y_s))

## Commented solution

In [ ]:
model = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=3000)).fit(X_s, y_s)
pred = model.predict(X_te)
cm = confusion_matrix(y_te, pred)
p, r, f, s = precision_recall_fscore_support(y_te, pred)
class_table = pd.DataFrame({"precision": p, "recall": r, "f1": f, "support": s})
class_table.round(3).T

In [ ]:
accuracy = accuracy_score(y_te, pred)
micro_f1 = f1_score(y_te, pred, average="micro")
macro_f1 = f1_score(y_te, pred, average="macro")
weighted_f1 = f1_score(y_te, pred, average="weighted")
micro_equals_accuracy = bool(np.isclose(micro_f1, accuracy))
print(f"accuracy {accuracy:.3f} | micro F1 {micro_f1:.3f} | macro {macro_f1:.3f} | weighted {weighted_f1:.3f}")

In [ ]:
silent_classes = [int(c) for c in class_table.index[(class_table["precision"] - class_table["recall"]) > 0.15]]
where_they_go = {}
for c in silent_classes:
    errors = cm[c].copy(); errors[c] = 0
    where_they_go[c] = int(np.argmax(errors))
print("classes with high precision and low recall:", silent_classes, "| their digits end up mostly in:", where_they_go)

In [ ]:
balanced = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=3000, class_weight="balanced")).fit(X_s, y_s)
pred_balanced = balanced.predict(X_te)
rare_recall_before = class_table.loc[[3, 8], "recall"].mean()
rare_recall_after = precision_recall_fscore_support(y_te, pred_balanced)[1][[3, 8]].mean()
balanced_macro_f1 = f1_score(y_te, pred_balanced, average="macro")
balanced_accuracy = accuracy_score(y_te, pred_balanced)
print(f"mean recall of the rare classes: {rare_recall_before:.3f} -> {rare_recall_after:.3f} | macro F1 {macro_f1:.3f} -> {balanced_macro_f1:.3f} | accuracy {accuracy:.3f} -> {balanced_accuracy:.3f}")

## Check

In [ ]:
with verify():
    check(cm.shape == (10, 10) and cm.sum() == len(y_te) and list(class_table.columns) == ["precision", "recall", "f1", "support"], "10x10 matrix and per-class table", "confusion_matrix(y_te, pred), precision_recall_fscore_support")
    check(close(accuracy, (pred == y_te).mean()) and close(macro_f1, f1_score(y_te, pred, average="macro")) and close(weighted_f1, f1_score(y_te, pred, average="weighted")), "correct accuracy, macro and weighted F1", "average='macro' and 'weighted'")
    check(micro_equals_accuracy is True and close(micro_f1, accuracy), "micro F1 = accuracy", "in multiclass with a single label per example, micro-F1 = accuracy")
    check(silent_classes == [3, 8], "the silent classes are 3 and 8: high precision, low recall", "precision - recall > 0.15")
    check(all(where_they_go[c] == int(np.argmax(np.where(np.eye(10, dtype=bool)[c], -1, cm[c]))) and where_they_go[c] != c for c in silent_classes), "for each silent class, the most frequent wrong class is given", "row of the class in the confusion matrix, without the diagonal")
    check(rare_recall_after > rare_recall_before + 0.05, "with class_weight='balanced' the recall of the rare classes goes up by over 0.05", "mean recall of classes 3 and 8")
    check(balanced_macro_f1 > macro_f1 and balanced_accuracy > accuracy, "and macro F1 and accuracy improve too", "compare the values before and after")
    check(macro_f1 < accuracy, "macro F1 is lower than the accuracy: the rare classes weigh more in the macro", "compare macro_f1 and accuracy")
done()

## What you learned

- The signature of the rare classes is **high precision, low recall**: the model predicts few of them, but when it does it is right. Looking only at the precision or the accuracy does not show it; the per-class recall shows it right away.
- The errors of the rare classes are not random: they end up in the "nearby" classes that the model has seen more often. The confusion matrix tells you where to intervene (more data, features to tell them apart) better than a number.
- **Micro F1 = accuracy** for single-label multiclass: it is a way of saying that micro counts the examples, not the classes. **The macro** weighs every class the same and for that reason drops with rare classes;
  the weighted one comes back close to the accuracy. Choose the average according to whom you care about (the rare classes or the total).
- `class_weight="balanced"` recovers recall on the rare classes and here it improves the overall accuracy too, because the model was wrong on two classes with few examples. It is not always so: often it is a compromise between
  the recall of the rare ones and the precision of the others (D2/B2).

## Common pitfall

Reporting the accuracy of a multiclass problem with imbalanced classes: the rare classes can have recall 0.5 and not move the total. Always report the per-class table.

## Interview question

> *Click on a question to show its answer.*

<details><summary>You have a macro F1 much lower than the accuracy. What does it tell you?</summary>

That there are (rare) classes on which the model does badly and that the accuracy hides them because they count little in the total. I look at precision and recall per class and the confusion matrix to see where the errors go, then
I evaluate interventions: more data or weights for the rare classes, features that tell them apart, different thresholds per class if the decisions allow it.

</details>